# Basic Usages of `viewer3d` library
##### Notebook author: Jason C. Klima

### Imports

In [ ]:
import attr
import ipywidgets
import pyrosetta
import pyrosetta.distributed.io as io
import time
import viewer3d as v3d

from bokeh.palettes import Greens256, Reds256, Viridis
from IPython.display import Image, display
from pyrosetta.rosetta.core.chemical import ResidueProperty
from pyrosetta.rosetta.core.select.residue_selector import (
    ChainSelector,
    LayerSelector,
    ResiduePropertySelector,
    SecondaryStructureSelector,
)
from pyrosetta.rosetta.core.simple_metrics.per_residue_metrics import (
    PerResidueEnergyMetric,
)
from pyrosetta.rosetta.protocols.backrub import BackrubMover
from pyrosetta.rosetta.protocols.minimization_packing import MinMover
from pyrosetta.rosetta.protocols.simple_moves import SmallMover

try:
    %load_ext jupyter_black
except ImportError:
    pass

### Set visualization backend used throughout the Jupyter notebook:
0) `"py3Dmol"`

1) `"nglview"`

2) `"pymol"`

### Set window size (width, height) used throughout the Jupyter notebook

In [ ]:
backend = "py3Dmol"
window_size = [1200, 600]


def on_backend_change(change):
    global backend
    if change["name"] == "value":
        backend = change["new"]


def on_width_change(change):
    global window_size
    if change["name"] == "value":
        window_size[0] = int(change["new"])


def on_height_change(change):
    global window_size
    if change["name"] == "value":
        window_size[1] = int(change["new"])


dropdown = ipywidgets.Dropdown(
    options=["py3Dmol", "nglview", "pymol"],
    value="py3Dmol",
    description="backend",
)
width_box = ipywidgets.IntText(
    value=1200,
    description="width",
)
height_box = ipywidgets.IntText(
    value=600,
    description="height",
)
dropdown.observe(on_backend_change)
width_box.observe(on_width_change)
height_box.observe(on_height_change)

display(dropdown, width_box, height_box)

### Initialize PyRosetta

In [ ]:
pyrosetta.init(
    options="-out:level 0",
    extra_options="",
    set_logging_handler="logging",
    silent=True,
)

### Visualize a PDB string (without instantiating a `Pose` object)

In [ ]:
pdb_id = "1QYS"  # Crystal structure of Top7: A computationally designed protein with a novel fold
pdbstring = v3d.pdbstring_from_pdb_id(pdb_id)

# For certain visualization modules and their keyword arguments, the input PDB string
# is sufficient and is not automatically converted to a `Pose` object.
v = v3d.init(
    pdbstring,
    window_size=window_size,
    backend=backend,
)
v += v3d.setBackgroundColor()
v += v3d.setStyle(
    residue_selector=None,
    cartoon=True,
    cartoon_color="darkgray" if backend != "pymol" else "grey30",
    cartoon_opacity=0.8,
    cartoon_radius=0.3,
    style="stick",
    colorscheme=(
        "amino"
        if backend == "py3Dmol"
        else "resname" if backend == "nglview" else "grey50"
    ),
    radius=0.2,
)
v += v3d.setStyle(
    command=({"resn": "HOH"}, {"sphere": {"color": "red", "radius": 0.3}})
)
v.show()

### Save a PNG file of a displayed visualization at any time

In [ ]:
# Certain arguments are accepted by different backends
# Also see `v.png` method to render a static image without saving it to disk
if backend == "py3Dmol":
    v.save_png("my_py3Dmol_figure.png")
elif backend == "nglview":
    v.save_png(
        "my_nglview_figure.png",
        factor=4,
        antialias=True,
        trim=False,
        transparent=False,
    )
elif backend == "pymol":
    v.save_png(
        "./my_pymol_figure.png",
        width=3000,
        height=2000,
        dpi=400,
        ray=True,
        ray_trace_mode=0,
        ray_shadows=False,
        opaque_background=True,
    )

### Visualize a `Pose` object

In [ ]:
pdb_id = "5BVL"  # Crystal structure of a de novo designed TIM-barrel
pose = v3d.pose_from_pdb_id(pdb_id)
v = v3d.init(
    pose,
    window_size=window_size,
    modules=[v3d.setBackgroundColor(), v3d.setStyle()],
    backend=backend,
)
v()  # Equivalent to v.show()

### Visualize multiple `Pose` objects

In [ ]:
poses = [v3d.pose_from_pdb_id(pdbid) for pdbid in ["6MSR", "1QCQ", "6J49"]]
v = (
    v3d.init(poses, window_size=window_size, backend=backend)
    + v3d.setBackgroundColor()
    + v3d.setStyle(
        colorscheme="lightgreyCarbon" if backend != "pymol" else "grey70"
    )
    + v3d.setHydrogenBonds()
    + v3d.setZoomTo()
)
v.show()

### Visualize selected residues of a `PackedPose` object

In [ ]:
packed_pose = io.to_packed(v3d.pose_from_pdb_id("2FD7"))
polar_residue_selector = ResiduePropertySelector(ResidueProperty(52))

v = v3d.init(packed_pose, window_size=window_size, backend=backend)
v.add(v3d.setBackgroundColor())
v.add(v3d.setStyle(radius=0.1))
v.add(
    v3d.setStyle(
        residue_selector=polar_residue_selector,
        colorscheme="whiteCarbon",
        radius=0.25,
        label=False,
    )
)
v.add(v3d.setHydrogens(color="grey", polar_only=True, radius=0.1))
v.add(v3d.setHydrogenBonds(color="black"))
v.add(v3d.setDisulfides(radius=0.1))
v()

### Visualize spheres

In [ ]:
v = sum(
    [
        v3d.init(packed_pose, window_size=window_size, backend=backend),
        v3d.setBackgroundColor(),
        v3d.setStyle(
            cartoon=False,
            style="sphere",  # See also: "stick", "cross", and "line"
            radius=1.5,
            colorscheme="darkgreyCarbon" if backend != "pymol" else "grey30",
        ),
        v3d.setZoom(factor=1.5),
    ]
)
v.show()

### Visualize chains

In [ ]:
pose = v3d.pose_from_pdb_id("6MSR")  # Crystal structure of pRO-2.5
chA = ChainSelector("A")
chB = ChainSelector("B")

v = sum(
    [
        v3d.init(pose, window_size=window_size, backend=backend),
        v3d.setBackgroundColor(),
        v3d.setStyle(
            cartoon_color="lightgrey" if backend != "pymol" else "grey70",
            radius=0.25,
        ),
        v3d.setSurface(
            residue_selector=chA,
            colorscheme="greenCarbon",
            opacity=0.65,
            surface_type="VDW" if backend != "pymol" else "SAS",
        ),
        v3d.setSurface(
            residue_selector=chB, color="blue", opacity=1.0, surface_type="SAS"
        ),
        v3d.setDisulfides(radius=0.25),
        v3d.setZoom(factor=1.25 if backend == "py3Dmol" else 0.1),
    ]
)
v()

### Visualize layers

In [ ]:
poses = [
    v3d.pose_from_alphafold_id("AF-A0A9W7HB40-F1", version="v6"),
    v3d.pose_from_alphafold_id("AF-Q96KD7-F1", version="v6"),
    v3d.pose_from_alphafold_id("AF-A0A1V3XXL8-F1", version="v6"),
]
core_selector = LayerSelector()
core_selector.set_layers(True, False, False)
boundary_selector = LayerSelector()
boundary_selector.set_layers(False, True, False)
surface_selector = LayerSelector()
surface_selector.set_layers(False, False, True)
v = (
    v3d.init(poses, window_size=window_size, backend=backend)
    + v3d.setStyle(
        residue_selector=core_selector,
        style="stick",
        radius=0.1,
        colorscheme="blackCarbon",
        label=False,
    )
    + v3d.setStyle(
        residue_selector=boundary_selector,
        style="stick",
        radius=0.3,
        colorscheme="whiteCarbon",
        label=True,
        label_fontsize=8,
        label_background=False,
        label_fontcolor="black",
    )
    + v3d.setStyle(
        residue_selector=surface_selector,
        style="stick",
        radius=0.1,
        colorscheme="blackCarbon",
        label=False,
    )
    + v3d.setSurface(
        opacity=0.5, surface_type="VDW" if backend != "pymol" else "SAS"
    )
    + v3d.setDisulfides()
    + v3d.setBackgroundColor(
        color="lightgrey" if backend != "pymol" else "grey70"
    )
    + v3d.setHydrogenBonds()
    + v3d.setHydrogens(polar_only=True, color="white")
    + v3d.setZoomTo(ChainSelector("A"))
)
v.show()

### Visualize secondary structure

In [ ]:
poses = [
    v3d.pose_from_alphafold_id("AF-A0A9W7HB40-F1", version="v6"),
    v3d.pose_from_alphafold_id("AF-Q96KD7-F1", version="v6"),
    v3d.pose_from_alphafold_id("AF-A0A1V3XXL8-F1", version="v6"),
]

helix_selector = SecondaryStructureSelector("H")
sheet_selector = SecondaryStructureSelector("E")
loop_selector = SecondaryStructureSelector("L")

modules = [
    v3d.setBackgroundColor(color="grey"),
    v3d.setStyle(
        residue_selector=helix_selector,
        cartoon=True,
        cartoon_color=0xFC0C7D,
        cartoon_radius=0.3,
        label=False,
        radius=0,
    ),
    v3d.setStyle(
        residue_selector=sheet_selector,
        cartoon=True,
        cartoon_color=0xFED10E,
        cartoon_radius=0.3,
        label=False,
        radius=0,
    ),
    v3d.setStyle(
        residue_selector=loop_selector,
        cartoon=True,
        cartoon_color="white",
        cartoon_radius=0.3,
        label=False,
        radius=0,
    ),
    v3d.setZoomTo(ChainSelector("A")),
]
if backend == "nglview":
    modules.append(
        v3d.setStyle(style="line", colorscheme="sstruc", cartoon=False)
    )

v = v3d.init(
    poses,
    window_size=window_size,
    modules=modules,
    continuous_update=True,
    backend=backend,
)
v()

In [ ]:
v.clear_modules()  # Subtract all visualization modules previously added to the `Viewer` object
v()

### Visualize a PyRosetta protocol in real-time (works best with `py3Dmol` and `pymol` backends)

In [ ]:
if backend == "nglview":
    print(
        "Warning: real-time trajectory visualizations work best with `py3Dmol` and `pymol` backends!"
    )
    time.sleep(3)

pose = v3d.pose_from_pdb_id("2FD7")
v = v3d.init(pose, delay=0.1, window_size=window_size, backend=backend)
v += v3d.setBackgroundColor(color="white")
v += v3d.setStyle(
    residue_selector=None,
    colorscheme="blackCarbon",
    radius=0.1,
    cartoon=True,
    cartoon_color="black",
    label=False,
)
v += v3d.setDisulfides(radius=0.1)
minimize = MinMover()
small = SmallMover()
small.nmoves(10)

v.show()
for _ in range(100):
    small.apply(pose)
    minimize.apply(pose)
    v.update_pose(pose)
    # Or simply use `v.update_viewer()` when the `Pose` object's memory address remains fixed

### Visualize the psi-space of residue #1 of a 20-residue polyvaline extended peptide

In [ ]:
n = 11  # 3, 4, 5, 6, 7, 8, 9, 10, 11, or 256
colors = Viridis[n]
v = v3d.init(window_size=window_size, backend=backend)
v()
pose = pyrosetta.io.pose_from_sequence("V" * 20)
for i, hex_str in enumerate(colors):
    v.set_modules(
        [
            v3d.setStyle(cartoon_color=hex_str, radius=0),
            v3d.setBackgroundColor(),
        ]
    )
    if i == 0:
        v += v3d.setZoomTo()
    pose.set_psi(1, i * 360 / n)
    v.add_pose(pose)

### Overlay multiple `Pose` objects

In [ ]:
pdb_id = "1U7I"  # Crystal Structure of Protein of Unknown Function PA1358 from Pseudomonas aeruginosa
pose1, pose2 = v3d.pose_from_pdb_id(pdb_id).split_by_chain()
v = v3d.init(pose1, window_size=window_size, backend=backend, auto_show=True)
v.set_modules(
    [
        v3d.setBackgroundColor(),
        v3d.setStyle(
            cartoon=True, cartoon_color="black", colorscheme="redCarbon"
        ),
        v3d.setSurface(
            color="red",
            opacity=0.5,
            surface_type="VDW" if backend != "pymol" else "SAS",
        ),
    ]
)
v.update_decoy(
    index=0
)  # Update `Viewer` object with current visualization modules
v.clear_modules()  # Subtract all visualization modules
v.set_modules(
    [
        v3d.setStyle(
            cartoon=True, cartoon_color="white", colorscheme="blueCarbon"
        ),
        v3d.setSurface(
            color="blue",
            opacity=0.5,
            surface_type="VDW" if backend != "pymol" else "SAS",
        ),
    ]
)
# Add `Pose` and update `Viewer` object with current visualization modules
v.add_pose(pose2)

### Visualize *SimpleMetrics* per-residue real metrics

In [ ]:
backrub = BackrubMover()
minimize = MinMover()
e = PerResidueEnergyMetric()
e.set_scorefunction(pyrosetta.create_score_function("ref2015"))
v = v3d.init(window_size=window_size, backend=backend, gui=True)
palette = list(Greens256) + list(reversed(Reds256))
v += v3d.setBackgroundColor(color="#dcdfe7")
v += v3d.setStyle(radius=0)
v += v3d.setPerResidueRealMetric(
    scoretype="res_energy",
    vmin=-5,
    vmax=5,
    radius=0.2,
    log=None,
    palette=palette,
    cartoon=True,
    cartoon_color="black",
    colorbar=True,
    colorbar_extremes=(True, True),
    colorbar_label="Per-residue Energy (ref2015)",
    colorbar_fontsize=16,
    colorbar_nticks=11,
    colorbar_discrete_ticks=True,
)
v += v3d.setHydrogens(
    polar_only=True, color="lightgray" if backend != "pymol" else "grey70"
)
v += v3d.setHydrogenBonds()
v += v3d.setDisulfides()


def run_protocol(pose):
    for _ in range(5):
        backrub.apply(pose)
        minimize.apply(pose)
    e.apply(pose)


pose = v3d.pose_from_pdb_id(
    "2FD7"
)  # X-ray Crystal Structure of Chemically Synthesized Crambin
for index in range(10):
    _pose = pose.clone()
    run_protocol(_pose)
    v.add_pose(_pose, index=index, update_viewer=False)
v.show()

### Change the color palette of an existing `Viewer` object

In [ ]:
poses = v.poses
modules = v.get_modules()
for i, module in enumerate(modules):
    if isinstance(module, v3d.setPerResidueRealMetric):
        params = attr.asdict(module, filter=lambda attrib, value: attrib.init)
        params["palette"] = v3d.get_matplotlib_cmap("coolwarm", num=256)
        modules[i] = v3d.setPerResidueRealMetric(**params)
v = v3d.init(modules=modules, window_size=window_size, backend=backend)
for index in poses:
    v.update_poses(poses[index], index=index, update_viewer=False)
v()

### Visualize different sets of different overlaid `Pose` objects

In [ ]:
pose = v3d.pose_from_pdb_id("2FD7")
v = v3d.init(pose, delay=0, window_size=window_size, backend=backend)
v.set_modules(
    [
        v3d.setBackgroundColor(),
        v3d.setStyle(),
        v3d.setDisulfides(),
        v3d.setZoomTo(),
    ]
)

backrub = BackrubMover()
minimize = MinMover()


def run_protocol(pose):
    for _ in range(5):
        backrub.apply(pose)
        minimize.apply(pose)


for index in range(2):
    for _ in range(3):  # Add 3 models per index
        run_protocol(pose)
        v.add_pose(pose.clone(), index=index, update_viewer=False)
v.show()

### Overlay all models from every index (updates the `Viewer` object above)

In [ ]:
v.overlay()